# OAuth2 — Browser Authorization-Code Flow (Approach C)

> **Design spec — integration-focused.** Companion to `2026-06-03-oauth2-refresh-grant-design.ipynb` (Approach B, **shipped**). This spec closes the one gap B left open: **acquiring the initial `refresh_token` in-app**, instead of asking the user to paste one obtained out-of-band.

| | |
|---|---|
| **Status** | Design / not yet planned |
| **Builds on** | Approach B — query-time refresh-token grant (`oauth.rs::access_token`, shipped) |
| **Crate** | `crates/spur-notebook/rest-table-gateway` + `crates/spur-notebook/src/mcp/tools/api_connection.rs` |
| **Author** | brain |
| **Date** | 2026-06-03 |

## The one-sentence problem

Approach B can **spend** a `refresh_token` forever (auto-minting access tokens at query time), but it cannot **acquire** one — today the wizard treats `<NAME>_REFRESH_TOKEN` as a plain missing env var and asks the user to paste it. Approach C runs the real OAuth2 **authorization-code** handshake (browser consent → loopback redirect → code exchange) so the token is obtained with one click and persisted for B to consume.

## Scope

**In scope**
- Parse the auth-code fields Nango already ships (`authorization_url`, `authorization_params`, `scope`/`scope_separator`, per-provider `connection_config`) — currently **dropped** at `ProviderEntry`.
- A second grant path in `oauth.rs`: build consent URL (with `state` + PKCE), exchange `grant_type=authorization_code`, capture the returned `refresh_token`.
- A loopback (`127.0.0.1:<port>/callback`) listener to catch `?code=…`.
- A persistence sink so the obtained `refresh_token` is visible to the query-time env lookup.
- A wizard branch: **\"Connect with browser\"** instead of **\"paste your refresh token.\"**

**Non-goals**
- The query-time runtime (`ManifestAdapter::scan` → `oauth::access_token`) is **unchanged** — once a `refresh_token` is persisted, Approach B takes over verbatim.
- OAUTH2_CC (client-credentials) and TWO_STEP / Plaid — separate tracks.
- Token encryption-at-rest beyond what the persistence sink already provides (called out as an open question, not solved here)."

## 1 · Integration map — where C plugs in

The system has a hard seam between **setup-time** (connect a data source, once) and **query-time** (run a table function, repeatedly). Approach C lives **entirely on the setup-time side**. The shaded `B` nodes already exist and do not change.

```mermaid
flowchart TB
    subgraph SETUP["⚙️  SETUP-TIME  (once per connection — Approach C)"]
        direction TB
        WIZ["call_add_api_connection<br/><i>api_connection.rs:184</i>"]
        REQ["required_env_vars_from_manifest<br/><i>api_connection.rs:476</i>"]
        BRANCH{"&lt;NAME&gt;_REFRESH_TOKEN<br/>missing?"}
        PASTE["TODAY: open_rest_wizard<br/>'paste your refresh token'"]
        CONNECT["NEW: 'Connect with browser'"]
        AUTHURL["build consent URL<br/>authorization_url + client_id<br/>+ redirect_uri + scope<br/>+ state + PKCE challenge"]
        BROWSER(["🌐 system browser<br/>provider consent screen"])
        LOOPBACK["NEW: loopback listener<br/>127.0.0.1:&lt;port&gt;/callback<br/>catches ?code=…&amp;state=…"]
        EXCHANGE["NEW: oauth::exchange_code<br/>grant_type=authorization_code<br/>→ refresh_token + access_token"]
        SINK[("NEW: credential sink<br/>persist refresh_token")]
    end

    subgraph PARSE["📦  MANIFEST TRANSLATION  (Nango → manifest)"]
        NANGO["ProviderEntry parse<br/><i>nango.rs:7</i><br/>NEW fields: authorization_url,<br/>authorization_params, scope"]
        AUTHCFG["AuthCfg::Oauth2Refresh<br/><i>manifest.rs:56</i><br/>(unchanged variant)"]
    end

    subgraph QUERY["🔁  QUERY-TIME  (every SELECT — Approach B, UNCHANGED)"]
        direction TB
        SCAN["ManifestAdapter::scan"]
        RESOLVE["resolve_auth → reads<br/>&lt;NAME&gt;_REFRESH_TOKEN from env"]
        ACCESS["oauth::access_token<br/><i>oauth.rs:44</i><br/>grant_type=refresh_token"]
        BEARER["apply_auth → Bearer header"]
        HTTP(["resource API<br/>e.g. api.notion.com"])
    end

    WIZ --> REQ --> BRANCH
    BRANCH -- "today" --> PASTE
    BRANCH -- "Approach C" --> CONNECT
    CONNECT --> AUTHURL --> BROWSER
    BROWSER -- "redirect" --> LOOPBACK
    LOOPBACK --> EXCHANGE --> SINK
    NANGO --> AUTHCFG
    AUTHURL -. "consumes parsed fields" .-> NANGO
    SINK == "refresh_token now resolvable" ==> RESOLVE
    PASTE == "manual paste" ==> RESOLVE
    SCAN --> RESOLVE --> ACCESS --> BEARER --> HTTP

    classDef shipped fill:#1f3a2e,stroke:#3fb27f,color:#cdebd9;
    classDef new fill:#3a2f1f,stroke:#d9a441,color:#f0e2c4;
    classDef browser fill:#2a2240,stroke:#9a7fd9,color:#e2d9f5;
    class SCAN,RESOLVE,ACCESS,BEARER,HTTP,AUTHCFG,PASTE,WIZ,REQ shipped;
    class CONNECT,AUTHURL,LOOPBACK,EXCHANGE,SINK,NANGO new;
    class BROWSER browser;
```

**Reading the diagram.** Green = ships today (B + existing wizard). Amber = new for C. Purple = the browser hop. The thick `==>` edge is the whole point: C's only job is to make `<NAME>_REFRESH_TOKEN` **resolvable**, by the same env lookup B already performs. Everything below the `RESOLVE` node is verbatim Approach B."

## 2 · The authorization-code handshake (sequence)

The interactive dance, end to end. Note the **two grants**: C performs `authorization_code` once at setup; B performs `refresh_token` on every later query. The `state` + PKCE pair is what makes the loopback redirect safe against interception and CSRF.

```mermaid
sequenceDiagram
    autonumber
    actor U as User
    participant W as MCP Wizard<br/>(api_connection.rs)
    participant L as Loopback Listener<br/>127.0.0.1:&lt;port&gt;
    participant B as System Browser
    participant P as Provider<br/>(authorization_url)
    participant T as Token Endpoint<br/>(token_url)
    participant S as Credential Sink

    Note over W: manifest = Oauth2Refresh, but<br/>&lt;NAME&gt;_REFRESH_TOKEN is missing
    U->>W: add_api_connection(notion)
    W-->>U: status: awaiting_credentials<br/>action: "Connect with browser"
    U->>W: click Connect

    W->>W: gen state (CSRF nonce)<br/>gen code_verifier + code_challenge (PKCE S256)
    W->>L: bind ephemeral port, start listener
    W->>B: open authorization_url?<br/>client_id, redirect_uri=127.0.0.1:port,<br/>scope, state, code_challenge
    B->>P: GET consent page
    P-->>U: show consent ("Notion wants access to…")
    U->>P: Approve
    P-->>B: 302 → 127.0.0.1:port/callback?code=…&state=…
    B->>L: GET /callback?code=…&state=…
    L->>L: verify state == issued nonce
    L-->>B: 200 "You can close this tab"

    L->>T: POST grant_type=authorization_code<br/>code, code_verifier, client_id,<br/>client_secret, redirect_uri
    T-->>L: { refresh_token, access_token, expires_in }
    L->>S: persist refresh_token (+ client_id/secret)
    S-->>W: <NAME>_REFRESH_TOKEN now resolvable
    W-->>U: status: ready ✓  (connection added)

    rect rgb(31,58,46)
    Note over W,T: ——— later, every query (Approach B, unchanged) ———
    W->>T: POST grant_type=refresh_token (auto, cached, skew 60s)
    T-->>W: { access_token }  → Bearer → resource API
    end
```

**Why each guard exists**
- **`state`** (step 7, verified step 13) — binds the redirect to *this* connect attempt; rejects forged callbacks.
- **PKCE `code_verifier`/`code_challenge`** (steps 7, 17) — proves the code-exchanger is the same agent that started the flow, even though the loopback redirect is observable to other local processes. Required for public-style clients; harmless for confidential ones.
- **Ephemeral loopback port** (step 6) — no fixed redirect URI to pre-register per machine; the listener lives only for the duration of the handshake and is torn down after step 17."

## 3 · Module DAG — new vs. changed

Graph-traced surface (from `code-explore`, 2026-06-03). Five touch-points; only **one** is a genuine design decision (the sink, ★).

```mermaid
flowchart LR
    subgraph existing["EXISTING — extend in place"]
        PE["ProviderEntry<br/><i>nango.rs:7-13</i>"]
        OA["oauth.rs::access_token<br/><i>refresh-token grant</i>"]
        TR["TokenResponse<br/><i>oauth.rs:31</i>"]
        WZ["call_add_api_connection<br/><i>api_connection.rs:184</i>"]
        REV["required_env_vars_from_manifest<br/><i>api_connection.rs:476</i>"]
    end

    subgraph new["NEW — add"]
        PARSE2["+ authorization_url<br/>+ authorization_params<br/>+ scope fields"]
        EXCH["oauth::exchange_code()<br/><i>grant_type=authorization_code</i>"]
        RTOK["+ refresh_token field<br/><i>capture from response</i>"]
        LOOP["oauth::loopback module<br/><i>bind + await ?code=</i>"]
        PKCE["oauth::pkce helpers<br/><i>verifier + S256 challenge + state</i>"]
        SINK["★ credential sink<br/><i>persist refresh_token</i>"]
        BR["wizard 'Connect with browser'<br/>branch"]
    end

    PE -->|gains| PARSE2
    TR -->|gains| RTOK
    OA -.->|sibling fn| EXCH
    EXCH --> RTOK
    EXCH --> PKCE
    EXCH --> LOOP
    WZ -->|new branch| BR
    BR --> EXCH
    EXCH --> SINK
    SINK -.->|read back by| REV

    classDef chg fill:#1d2b3a,stroke:#4f8fd9,color:#cfe3f7;
    classDef add fill:#3a2f1f,stroke:#d9a441,color:#f0e2c4;
    classDef star fill:#3a1f2f,stroke:#d9418f,color:#f7cfe3;
    class PE,OA,TR,WZ,REV chg;
    class PARSE2,EXCH,RTOK,LOOP,PKCE,BR add;
    class SINK star;
```

### Touch-point table

| # | File | Change | Risk |
|---|------|--------|------|
| 1 | `nango.rs:7` `ProviderEntry` | Add `authorization_url`, `authorization_params`, `scope` (Optional) — fields exist in snapshot YAML, currently dropped | Low — additive serde, same pattern as Gap 2/5 |
| 2 | `oauth.rs` `TokenResponse:31` | Add `refresh_token: Option<String>` (currently only `access_token`/`expires_in`) | Low |
| 3 | `oauth.rs` | New `exchange_code()` sibling to `access_token()`; new `pkce`/`state` helpers | Med — new crypto-adjacent code, needs tests |
| 4 | `oauth.rs` | New `loopback` listener (bind ephemeral, await one request, parse `code`/`state`) | Med — async + port lifecycle |
| 5 | `api_connection.rs:184` `call_add_api_connection` | New "Connect with browser" branch when `Oauth2Refresh` + token missing | Med — wizard UX + orchestration |
| 6 ★ | **credential sink** (location TBD) | Persist obtained `refresh_token` so query-time env lookup resolves it | **High — the real design decision** (see §6) |

**The ★ is the crux.** Everything else mirrors an existing pattern. The sink is the only choice that ripples into security, multi-machine portability, and the global-connections model — it gets its own section."

## 4 · Connection lifecycle (state machine)

The states the wizard moves a connection through, and the recovery edges. C owns the amber band (`NeedsAuth` → `Connected`); B owns the green band (`Connected` ⇄ `Refreshing`).

```mermaid
stateDiagram-v2
    direction TB
    [*] --> Resolving: add_api_connection

    Resolving --> Connected: token already in env<br/>(B-only path)
    Resolving --> NeedsAuth: Oauth2Refresh &amp;&amp;<br/>refresh_token missing

    state "Approach C — acquire" as C {
        NeedsAuth --> AwaitingConsent: click 'Connect with browser'<br/>(listener bound, browser opened)
        AwaitingConsent --> ExchangingCode: ?code=…&amp;state=… received<br/>(state verified)
        AwaitingConsent --> NeedsAuth: user cancels / tab closed
        AwaitingConsent --> AuthError: state mismatch / timeout
        ExchangingCode --> Persisting: token_url returns refresh_token
        ExchangingCode --> AuthError: exchange rejected (4xx)
        Persisting --> Connected: refresh_token written to sink
    }

    state "Approach B — spend (shipped)" as B {
        Connected --> Refreshing: query &amp;&amp; access token<br/>within 60s of expiry
        Refreshing --> Connected: new access_token cached
        Refreshing --> Reauth: refresh_token revoked/expired<br/>(token_url 4xx)
    }

    AuthError --> NeedsAuth: retry
    Reauth --> NeedsAuth: re-run 'Connect with browser'
    Connected --> [*]: connection removed

    note right of C
        C runs ONCE per connection.
        Ephemeral port + state + PKCE
        live only inside AwaitingConsent
        → ExchangingCode.
    end note
    note right of B
        Unchanged from the shipped
        refresh-grant spec. Reauth is
        the ONLY new edge B gains —
        it routes a dead refresh_token
        back into C instead of erroring.
    end note
```

**The one new edge into B.** Today, a revoked/expired `refresh_token` surfaces as a `GatewayError::Auth` at query time and dead-ends. With C present, `Refreshing --> Reauth --> NeedsAuth` becomes a real recovery loop: the user re-clicks "Connect with browser" and the connection heals without being deleted and recreated. That edge is the second-order payoff of building C — not just first-token acquisition, but **self-healing re-consent**."

## 5 · UX — the "Connect with browser" wizard

**Surface:** the REST API connection wizard (a panel inside the notebook host shell).
**Direction:** *dark dev-tool* — same palette as the shipped refresh-grant spec, so C reads as a continuation, not a new product. Mono numerics, low-chroma surfaces, one decisive accent (provider-tinted) on the primary action.

### The UX problem C solves
The paste-a-refresh-token flow asks the user to leave the app, find a developer console, run a one-off OAuth tool, copy an opaque string, and paste it — five context switches and a "what even is a refresh token?" cliff. C collapses that to **one button and one browser tab the user already trusts.**

### Three states the user actually sees
1. **Ready to connect** — provider chosen, the wizard says *"This connection signs in with your Notion account"* and offers one primary button: **Connect with browser**. No token field. The three secrets (`client_id`/`client_secret` for the app, plus the acquired token) are framed as "handled for you" except where the user genuinely must supply an app registration.
2. **Waiting for consent** — button becomes a calm spinner: *"Finishing in your browser… approve access to continue."* A secondary *Cancel* tears down the listener. This is the only state with a time pressure, so it must feel patient, not anxious.
3. **Connected** — a single green confirmation with the concrete payoff: the **table functions now callable** (`notion_pages()`, `notion_databases()`), so the user immediately sees *why* they connected, not just *that* they did.

### Anti-slop guards applied
- No generic "Success!" checkmark with no content — the success state names the **actual table functions** the user unlocked.
- No fake progress bar — the consent step is genuinely indeterminate, so it shows an honest spinner with a real instruction, not a lying 0→100%.
- The error state names the **specific failure** (`state mismatch`, `consent denied`, `app not registered`) with the matching recovery, never "Something went wrong."

The cell below renders the interactive mock (`text/html`)."

In [2]:
# open-design artifact — "Connect with browser" wizard (Approach C)
from IPython.display import HTML

html = """
<!DOCTYPE html><html><head><meta charset="utf-8"><style>
  :root{
    --bg:#0e1116; --panel:#161b22; --panel2:#1b212b; --line:#2a323d;
    --txt:#c9d4e0; --muted:#7d8a99; --mono:'SF Mono',ui-monospace,Menlo,monospace;
    --grn:#3fb27f; --grn-bg:#16271f; --amb:#d9a441; --amb-bg:#241d10;
    --pur:#9a7fd9; --dgr:#e0556b; --dgr-bg:#2a1419;
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--bg);font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
       color:var(--txt);padding:28px;display:flex;justify-content:center}
  .wrap{width:520px}
  .switch{display:flex;gap:6px;margin-bottom:16px}
  .switch button{flex:1;background:var(--panel);border:1px solid var(--line);color:var(--muted);
       font-size:11px;letter-spacing:.04em;text-transform:uppercase;padding:7px 0;border-radius:6px;cursor:pointer}
  .switch button.on{color:var(--txt);border-color:#3d4b5c;background:var(--panel2)}
  .card{background:var(--panel);border:1px solid var(--line);border-radius:12px;overflow:hidden;
        box-shadow:0 12px 40px rgba(0,0,0,.45)}
  .hd{display:flex;align-items:center;gap:10px;padding:16px 18px;border-bottom:1px solid var(--line)}
  .hd .t{font-size:13px;font-weight:600;letter-spacing:.01em}
  .hd .sub{font-size:11px;color:var(--muted)}
  .chip{margin-left:auto;display:flex;align-items:center;gap:7px;background:var(--panel2);
        border:1px solid var(--line);border-radius:20px;padding:4px 11px 4px 5px;font-size:12px}
  .chip .dot{width:20px;height:20px;border-radius:6px;background:#e9e9ea;color:#111;display:grid;
        place-items:center;font-weight:800;font-size:12px}
  .body{padding:22px 18px 24px}
  .lead{font-size:14px;line-height:1.55}
  .lead b{color:#eef3f9}
  .muted{color:var(--muted);font-size:12.5px;line-height:1.5;margin-top:8px}
  .btn{margin-top:20px;width:100%;border:none;border-radius:9px;padding:13px;font-size:14px;
       font-weight:600;cursor:pointer;display:flex;align-items:center;justify-content:center;gap:9px}
  .btn-primary{background:linear-gradient(180deg,#eef2f7,#d7dee7);color:#10151c}
  .btn-ghost{margin-top:12px;background:transparent;border:1px solid var(--line);color:var(--muted)}
  .row{display:flex;gap:9px;align-items:flex-start;margin-top:14px}
  .ic{width:22px;height:22px;border-radius:6px;flex:none;display:grid;place-items:center;font-size:12px;font-weight:700;margin-top:1px}
  .spin{width:17px;height:17px;border-radius:50%;border:2.5px solid #3a4654;border-top-color:#eef2f7;
        animation:sp .8s linear infinite}
  @keyframes sp{to{transform:rotate(360deg)}}
  .fn{font-family:var(--mono);font-size:12.5px;background:#0c1117;border:1px solid var(--line);
      border-radius:6px;padding:9px 11px;margin-top:8px;color:#a9d6c2;display:flex;justify-content:space-between}
  .fn .ok{color:var(--grn)}
  .banner{border-radius:9px;padding:12px 14px;font-size:13px;line-height:1.5;display:flex;gap:10px;align-items:flex-start}
  .b-grn{background:var(--grn-bg);border:1px solid #265c45;color:#bdebd6}
  .b-amb{background:var(--amb-bg);border:1px solid #5c4a1f;color:#f0dba8}
  .b-dgr{background:var(--dgr-bg);border:1px solid #5c2630;color:#f3c0c8}
  .label{font-size:10.5px;letter-spacing:.08em;text-transform:uppercase;color:var(--muted);margin:18px 0 7px}
  .hide{display:none}
  .foot{padding:11px 18px;border-top:1px solid var(--line);font-size:11px;color:var(--muted);
        display:flex;justify-content:space-between;font-family:var(--mono)}
</style></head><body><div class="wrap">

  <div class="switch">
    <button class="on" onclick="show('ready',this)">1 · Ready</button>
    <button onclick="show('wait',this)">2 · Consent</button>
    <button onclick="show('done',this)">3 · Connected</button>
    <button onclick="show('err',this)">Error</button>
  </div>

  <div class="card">
    <div class="hd">
      <div>
        <div class="t">Add REST connection</div>
        <div class="sub">OAuth2 · authorization-code</div>
      </div>
      <div class="chip"><span class="dot">N</span>Notion</div>
    </div>
    <div class="body">

      <!-- READY -->
      <div id="ready">
        <div class="lead">This connection signs in with your <b>Notion</b> account. No tokens to copy &mdash; approve once in your browser and we keep it fresh for you.</div>
        <div class="muted">We&rsquo;ll open Notion&rsquo;s consent page, capture the grant on a local <span style="font-family:var(--mono)">127.0.0.1</span> redirect, and store a refresh token scoped to this connection.</div>
        <button class="btn btn-primary" onclick="show('wait',document.querySelectorAll('.switch button')[1])">
          <span style="font-size:15px">&#9658;</span> Connect with browser
        </button>
        <div class="label">App registration (advanced)</div>
        <div class="muted" style="margin-top:0">Using SPUR&rsquo;s shared Notion app. Override with your own <span style="font-family:var(--mono)">client_id</span> / <span style="font-family:var(--mono)">client_secret</span> only if you need a private integration.</div>
      </div>

      <!-- WAITING -->
      <div id="wait" class="hide">
        <div class="banner b-amb"><div class="spin"></div>
          <div><b>Finishing in your browser.</b><br>Approve access to Notion in the tab we just opened. This page updates automatically.</div></div>
        <div class="row"><div class="ic" style="background:#23303f;color:var(--pur)">&#9737;</div>
          <div class="muted" style="margin:0">Listening on <span style="font-family:var(--mono)">127.0.0.1:51847/callback</span> &mdash; state &amp; PKCE verified on return.</div></div>
        <button class="btn btn-ghost" onclick="show('ready',document.querySelectorAll('.switch button')[0])">Cancel</button>
      </div>

      <!-- DONE -->
      <div id="done" class="hide">
        <div class="banner b-grn"><span style="font-size:15px">&#10003;</span>
          <div><b>Notion connected.</b> Refresh token stored &mdash; queries auto-refresh, no re-auth needed.</div></div>
        <div class="label">Callable now</div>
        <div class="fn"><span>notion_pages()</span><span class="ok">ready</span></div>
        <div class="fn"><span>notion_databases()</span><span class="ok">ready</span></div>
        <div class="fn"><span>notion_users()</span><span class="ok">ready</span></div>
        <button class="btn btn-primary" style="margin-top:18px">Done</button>
      </div>

      <!-- ERROR -->
      <div id="err" class="hide">
        <div class="banner b-dgr"><span style="font-size:15px">&#9888;</span>
          <div><b>Consent was denied.</b> Notion reported the access request was declined &mdash; nothing was stored.</div></div>
        <div class="muted">Other specific failures this state names instead of &ldquo;something went wrong&rdquo;:</div>
        <div class="row"><div class="ic" style="background:var(--dgr-bg);color:var(--dgr)">!</div>
          <div class="muted" style="margin:0"><b style="color:#f3c0c8">state mismatch</b> &mdash; the redirect didn&rsquo;t match this attempt; ignored for safety. Retry.</div></div>
        <div class="row"><div class="ic" style="background:var(--dgr-bg);color:var(--dgr)">!</div>
          <div class="muted" style="margin:0"><b style="color:#f3c0c8">app not registered</b> &mdash; add a <span style="font-family:var(--mono)">client_id</span>/<span style="font-family:var(--mono)">client_secret</span> for a private integration.</div></div>
        <button class="btn btn-primary" onclick="show('ready',document.querySelectorAll('.switch button')[0])" style="margin-top:18px">Try again</button>
      </div>

    </div>
    <div class="foot"><span>scheme: oauth2_refresh</span><span>grant: authorization_code &rarr; refresh_token</span></div>
  </div>
</div>
<script>
  function show(id,btn){
    ['ready','wait','done','err'].forEach(function(x){document.getElementById(x).classList.add('hide')});
    document.getElementById(id).classList.remove('hide');
    document.querySelectorAll('.switch button').forEach(function(b){b.classList.remove('on')});
    if(btn) btn.classList.add('on');
  }
</script>
</body></html>
"""
HTML(html)

## 6 · The ★ decision — where does the `refresh_token` live?

This is the only choice that isn't mechanical. Three options, scored against the existing env-var contract that Approach B reads from.

| Option | How B reads it | Portable across machines | Secure-at-rest | Verdict |
|--------|----------------|--------------------------|----------------|---------|
| **A · process env var** (write `<NAME>_REFRESH_TOKEN` into the daemon's env) | already the B path | ✗ lost on restart | ✗ plaintext in process | reject — ephemeral |
| **B · connection store file** (extend the saved-connection record at `~/.spur/connections.json`) | resolver reads file → env shim | ✓ survives restart | △ file perms only | **recommended for v1** |
| **C · OS keychain** (Keychain / Secret Service) | resolver queries keychain | ✓ | ✓ | best, but new dep + per-OS code — defer to v2 |

**Recommendation:** ship **Option B** (connection store), with the resolver shim that maps a stored token into the `<NAME>_REFRESH_TOKEN` lookup B already performs. Keep the sink behind a trait so **Option C** can replace it without touching `exchange_code` or B. This mirrors the existing global-reusable-connections design (`2026-06-01-global-reusable-api-connections-design.md`).

> **Open question for review:** does the connection store already persist *credentials*, or only *manifests*? If only manifests, the sink is genuinely new surface and needs its own threat review. Resolve before planning.

## 7 · Integration contract (the seam guarantees)

For C to be a drop-in that doesn't perturb B:
1. **C writes, B reads, through one key.** C's only output is making `<NAME>_REFRESH_TOKEN` resolvable. No new query-time code path.
2. **`exchange_code` and `access_token` share `TokenResponse`.** Add `refresh_token: Option<String>` once; both grants deserialize the same struct.
3. **The loopback listener is setup-only.** It binds, serves exactly one request, and is dropped before `call_add_api_connection` returns. No long-lived port.
4. **Parsing is additive.** New `ProviderEntry` fields are `Option<_>`; providers without `authorization_url` (the SAMPLE, OAUTH2_CC, TWO_STEP) keep their current behavior — no `Oauth2Refresh` auth-code branch is taken.
5. **Cross-crate exhaustiveness.** No new `AuthCfg` variant is required (C reuses `Oauth2Refresh`), so the E0004 trap that bit Approach B at merge does **not** recur. Confirm with `cargo check -p spur-notebook` regardless.

## 8 · Security model
- **PKCE S256** on every exchange (verifier never leaves the process; only the challenge goes to the provider).
- **`state` nonce** generated per attempt, verified on the redirect, single-use.
- **Loopback only** — `redirect_uri` is always `127.0.0.1:<ephemeral>`; never a public URL. Reject any callback whose `Host` isn't loopback.
- **`client_secret`** never reaches the browser — it is used only in the back-channel `token_url` POST (steps 17 of §2).
- **Stored token** inherits the sink's protection (§6); v1 = file perms, v2 = keychain.

## 9 · Acceptance criteria
- [ ] A Nango provider with `authorization_url` round-trips through `ProviderEntry` (new failing test → green).
- [ ] `oauth::exchange_code` POSTs `grant_type=authorization_code` with `code_verifier` and returns `{refresh_token, access_token}` (wiremock test, mirrors the existing `access_token` tests).
- [ ] `state` mismatch and non-2xx exchange both yield `GatewayError::Auth` (never panic, never silent success).
- [ ] Loopback listener binds an ephemeral port, resolves on one `?code=` request, and is torn down.
- [ ] After a simulated exchange, `required_env_vars_from_manifest`'s `<NAME>_REFRESH_TOKEN` resolves from the sink — i.e. a connection that was `awaiting_credentials` flips to `ready` without a manual paste.
- [ ] `cargo test -p spur-rest-table-gateway` and `cargo check -p spur-notebook` both green.

## 10 · Build sequence (for a future `writing-plans` spec)
1. **T1** — `ProviderEntry` parse + `TokenResponse.refresh_token` (additive, isolated).
2. **T2** — `oauth::pkce` + `oauth::loopback` (pure, unit-testable, no wizard).
3. **T3** — `oauth::exchange_code` wiring T1+T2 (wiremock).
4. **T4 ★** — credential sink behind a trait (the design-review gate).
5. **T5** — `call_add_api_connection` "Connect with browser" branch (depends on all).

> **Not planned yet.** This notebook is the design artifact only — submit to `writing-plans` → `submit_plan` after the §6 open question is resolved in review."